# Reproducing Campbell et al. 2021 (WHO 2021 reference distribution)

Independent reproduction of *Distribution of semen examination results 2020 - A
follow up of data collated for the WHO semen analysis manual 2010* (Andrology
2021;9:817-822, doi:10.1111/andr.12983) from the authors' public deposit
`Fertile_Man_2020.xlsx` (University of Dundee, doi:10.15132/10000163, CC BY),
checking the table that `seminal.semen` carries.

Sections:

1. The reference men: studies and completeness
2. Table 3 (WHO 2020): recompute the nine centiles
3. Table 2 (2010 to 2020 studies): which studies, and the centiles
4. Uncertainty on the 5th centile
5. Heterogeneity between contributing studies
6. How good is the interpolated percentile?


## 1. The reference men

`make data_gen preprocess` saves the deposit as `data/processed/fertile_man_2020.parquet`
with the nine parameters renamed to the library's names and `NO RESULT` set to
missing. Nothing is excluded: the paper used every value present, so the count
of non-missing values per parameter should match the `n` column of Table 3.

In [ ]:
import numpy as np
import pandas as pd

from seminal import semen_reference
from seminal.semen import CENTILES, PARAMETERS

fm = pd.read_parquet("../data/processed/fertile_man_2020.parquet")
params = list(PARAMETERS)

print(f"{len(fm):,} men from {fm['study'].nunique()} studies")
print(fm["study"].value_counts().to_string())

# non-missing per parameter next to the paper's n
pd.DataFrame({"n_deposit": fm[params].notna().sum(), "n_paper": semen_reference()["n"]})

## 2. Table 3: recompute the nine centiles

The paper does not state its quantile definition, and with discrete, heavily
tied parameters the definition matters at the printed precision. Trying the
standard ones against the printed table singles out the Weibull definition
(Hyndman and Fan type 6, what SPSS and Excel's `PERCENTILE.EXC` use), with
ordinary round-half-up to the printed precision: volume to one decimal,
everything else to whole numbers. Compare the recomputed centiles with the
library's copy of Table 3.

In [ ]:
from decimal import ROUND_HALF_UP, Decimal

Q = [c / 100 for c in CENTILES]
cols = [f"p{c:g}" for c in CENTILES]

def round_half_up(x, decimals):
    q = Decimal("0.1") if decimals else Decimal("1")
    return float(Decimal(str(float(x))).quantize(q, rounding=ROUND_HALF_UP))

def centile_table(df, method="weibull", rounded=True):
    """Nine centiles per parameter, rounded as the paper prints them."""
    out = pd.DataFrame({p: np.quantile(df[p].dropna(), Q, method=method) for p in params}, index=cols).T
    if rounded:
        for p in params:
            out.loc[p] = [round_half_up(v, 1 if p == "volume" else 0) for v in out.loc[p]]
    return out

empirical = centile_table(fm)
published = semen_reference()[cols]

diff = empirical - published
print("cells that differ from Table 3:", int((diff != 0).sum().sum()), "of", diff.size)
diff[diff != 0].dropna(how="all").dropna(axis=1, how="all")

One cell in 81: the median total sperm number comes out at 209.496, which the
paper prints as 210 (displayed as 209.5, then rounded). Everything else,
including all nine 5th centiles that the WHO manual uses as lower reference
limits, matches exactly. With numpy's default linear interpolation six cells
differ by a unit instead, all at the extreme centiles of the discrete parameters.

In [ ]:
side_by_side = pd.concat({"deposit": empirical, "paper": published}, axis=1).swaplevel(axis=1).sort_index(axis=1)
side_by_side[[f"p{c:g}" for c in (5, 50, 95)]]

## 3. Table 2: the 2010 to 2020 studies

Table 2 is the subset of studies published after the 2010 manual, n = 1,789
for most parameters. The deposit does not label which studies those are, but
only one combination of the ten study sizes sums to 1,789: Tang, Zedan,
Aboutorabi, Lotti and Evgeni. The other five (Auger, Swan, Baker, Jensen,
Haugen, n = 1,800) are the cohorts behind the 2010 manual.

In [ ]:
counts = fm["study"].value_counts()
new_studies = ["Tang", "Zedan", "Aboutorabi", "Lotti", "Evgeni"]
print("2010-2020 studies:", counts[new_studies].sum(), " 2010 cohorts:", counts.drop(new_studies).sum())

sub = fm[fm["study"].isin(new_studies)]
empirical_2 = centile_table(sub)
published_2 = semen_reference(edition="2010_2020")[cols]
diff_2 = empirical_2 - published_2
print("cells that differ from Table 2:", int((diff_2 != 0).sum().sum()), "of", diff_2.size)
diff_2[diff_2 != 0].dropna(how="all").dropna(axis=1, how="all")

Table 2 reproduces cell for cell with the same definition.

## 4. Uncertainty on the 5th centile

The paper gives a 95% CI for each 5th centile. Bootstrap the deposit and
compare. The CI is what separates "16" from "somewhere in 15 to 18" when a lab
reports a concentration of 15.

In [ ]:
rng = np.random.default_rng(0)
B = 2000

def boot_p5(s):
    s = s.to_numpy()
    draws = rng.choice(s, size=(B, len(s)), replace=True)
    return np.percentile(np.quantile(draws, 0.05, axis=1, method="weibull"), [2.5, 97.5])

ci = pd.DataFrame({p: boot_p5(fm[p].dropna()) for p in params}, index=["boot_low", "boot_high"]).T
ci.insert(0, "p5", published["p5"])
ci["paper_low"] = semen_reference()["p5_ci_low"]
ci["paper_high"] = semen_reference()["p5_ci_high"]
ci.round(2)

The bootstrap intervals sit on or inside the published ones. The published CI
for normal forms (3.9 to 4.0) is tighter than a bootstrap gives because
morphology is reported in whole percent and the 5th centile sits on a plateau
of 4s.

## 5. Heterogeneity between studies

Paffoni et al. (2022) argue the ten studies are not one population. The
simplest view is the median of each parameter by study. Differences in
motility and morphology between laboratories are much larger than between
countries' men, which is the point: the reference distribution describes the
pooled practice of these laboratories as much as fertile men.

In [ ]:
import matplotlib.pyplot as plt

by_study = fm.groupby("study")[params].median().loc[counts.index]
by_study["n"] = counts

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, p in zip(axes.ravel(), params):
    data = [fm.loc[fm["study"] == s, p].dropna() for s in counts.index]
    ax.boxplot(data, tick_labels=counts.index, showfliers=False)
    ax.axhline(published.loc[p, "p5"], color="#C0392B", linestyle="--", linewidth=1)
    ax.set_title(PARAMETERS[p], fontsize=10)
    ax.tick_params(axis="x", rotation=60, labelsize=8)
fig.suptitle("Fertile men by contributing study (dashed line: WHO 2021 lower reference limit)", y=1.0)
fig.tight_layout()
plt.show()

by_study.round(1)

Two things stand out. Immotile and vitality are missing for whole studies
(hence their smaller n in the table), and the morphology medians range from
about 5% to over 20% between laboratories, a spread no biological difference
between fertile men explains. A lab comparing its own patients to these
centiles is comparing against a mixture.

## 6. How good is the interpolated percentile?

`semen_percentile` interpolates between the nine published centiles because
the full distribution is not published. With the deposit in hand, compare it
to the empirical percentile of every man (his rank among the reference men).

In [ ]:
from seminal import semen_percentile

rows = []
for p in params:
    s = fm[p].dropna().to_numpy()
    emp = pd.Series(s).rank(pct=True).to_numpy() * 100            # empirical percentile
    lib = semen_percentile(p, s)                                  # interpolated from Table 3
    inside = (emp >= 2.5) & (emp <= 97.5)                          # where interpolation applies
    err = np.abs(lib - emp)[inside]
    rows.append({"parameter": p, "n": len(s), "median_abs_err": np.median(err),
                 "p95_abs_err": np.percentile(err, 95), "max_abs_err": err.max()})
pd.DataFrame(rows).set_index("parameter").round(2)

Inside the published range the interpolated percentile is typically within a
point or two of the empirical one; the largest errors are on the discrete,
heavily tied parameters (non-progressive motility, normal forms) between the
25th and 75th centiles, where the published grid is coarsest. For a clinical
read ("about the 30th centile") that is adequate; for anything finer, use the
deposit directly.

## 7. Scoring a cohort with `add_semen_percentile_cols`

Everything above recomputes the table. In practice you have a cohort and want
to know where each man sits against the fertile-man reference.
`add_semen_percentile_cols` takes a DataFrame and a mapping of parameter name
to column name (`{"concentration": "conc", ...}`) and adds, per parameter, a
percentile column (`_pct`, 2.5 to 97.5) and a below-limit flag
(`_below_lrl`, below the WHO 5th centile). Here the deposit's columns already
carry the parameter names, so the mapping can be omitted.

Scoring the reference men is also a self-check: by construction about 5% of
them fall below each limit.

In [ ]:
from seminal import add_semen_percentile_cols

scored = add_semen_percentile_cols(fm)

# share below the lower reference limit, per parameter (expect ~5%)
flags = [f"{p}_below_lrl" for p in params]
below = scored[flags].mean().rename(lambda c: c.replace("_below_lrl", "")) * 100
pd.DataFrame({"n": fm[params].notna().sum(), "% below LRL": below.round(1)})

The pooled rate is 5% because the limits were derived from these men. Per
study it is not, which is the heterogeneity of section 5 in the units a clinic
would see: the share of its fertile men that a WHO-based report would flag. Note the direction for `immotile`: a man flags when
his immotile fraction is below the 5th centile, which is the favourable end,
so that column reads as "unusually few immotile sperm" rather than as a
deficit. Aboutorabi's 40% there is a scoring convention, not biology.

In [ ]:
by_study_flag = scored.groupby("study")[flags].mean().loc[counts.index] * 100
by_study_flag.columns = [c.replace("_below_lrl", "") for c in flags]
by_study_flag["n"] = counts
by_study_flag.round(1)

Three things decide whether the scored columns mean anything on another
dataset:

* **Units** must match `PARAMETERS[name]["unit"]`: mL, 10^6/mL, 10^6 per
  ejaculate, and percentages on a 0 to 100 scale. A motility column stored as a
  fraction (0.55) scores as the 2.5th centile; multiply by 100 first. If total
  sperm number is not recorded, compute `volume * concentration`.
* **Morphology** is only comparable if the laboratory scored strict (Tygerberg)
  criteria, the method behind the reference. Older WHO criteria give values
  several times higher, and a 30% result would wrongly land at the top of the
  distribution.
* **Missing or unparseable** values pass through as `NaN` (percentile) and
  `<NA>` (flag). Do not fill them before scoring.

With a mapping, the call for a clinic export with its own column names looks
like this:

In [ ]:
clinic = pd.DataFrame(
    {
        "pid": ["A", "B", "C", "D"],
        "vol_ml": [3.2, 1.1, 2.5, None],
        "conc_m_per_ml": [45, 8, 120, 60],
        "pr_pct": [40, 20, 65, 0.55],   # D was exported as a fraction
        "strict_morph_pct": [5, 2, 9, 4],
    }
)
clinic["pr_pct"] = clinic["pr_pct"].where(clinic["pr_pct"] > 1, clinic["pr_pct"] * 100)
clinic["tsc_m"] = clinic["vol_ml"] * clinic["conc_m_per_ml"]

scored_clinic = add_semen_percentile_cols(
    clinic,
    columns={
        "volume": "vol_ml",
        "concentration": "conc_m_per_ml",
        "total_number": "tsc_m",
        "progressive_motility": "pr_pct",
        "normal_forms": "strict_morph_pct",
    },
)
scored_clinic["n_below_lrl"] = scored_clinic.filter(like="_below_lrl").sum(axis=1)
scored_clinic

`n_below_lrl` (0 to 9) is a simple severity count. The percentiles are a
monotone transform of the raw values, so they add nothing to a tree model, but
they put the nine parameters on one scale for plotting, for linear models and
for comparing a cohort against the reference.

## Summary

- The deposit contains exactly the men behind Table 3: non-missing counts
  match the paper's n for all nine parameters.
- All 81 cells of Table 3 reproduce from the raw data with the Weibull
  (SPSS / Excel `PERCENTILE.EXC`) quantile definition, except the median total
  sperm number, which the paper rounds from 209.5 to 210. The 5th centiles,
  the WHO 2021 lower reference limits, reproduce exactly.
- Table 2 (2010 to 2020 studies) is the Tang, Zedan, Aboutorabi, Lotti and
  Evgeni cohorts and reproduces cell for cell.
- Bootstrap CIs on the 5th centile agree with the published CIs.
- The pooled distribution hides large between-laboratory differences,
  especially in morphology; the centiles are WHO's, and `seminal` reports them
  as such.
- `semen_percentile` is exact at the published centiles and within a few
  points elsewhere.
- `add_semen_percentile_cols` scores any cohort against the reference; on the
  reference men themselves 5% flag below each limit overall, with wide
  variation by study.
